# Macro Nowcaster: walkthrough
End-to-end demo. Runs on synthetic data with no key; set `FRED_API_KEY` for live data and `ANTHROPIC_API_KEY` for the Fed RAG analysis and written memos.

In [ ]:
from macro_nowcaster.pipeline import build_artifact
art = build_artifact(persist=False)
import json; print(json.dumps(art.summary(), indent=2))

## Composite activity index and recession probability

In [ ]:
import plotly.graph_objects as go
f = art.activity.factor
fig = go.Figure()
fig.add_scatter(x=f.index, y=f.values, name='composite', line=dict(color='#2166ac'))
fig.add_hline(y=0, line_dash='dash')
fig.update_layout(title=f'Composite activity ({art.activity.method}, var explained {art.activity.var_explained:.0%})', height=320)
fig.show()

## Honest evaluation: pseudo-real-time replay
Real-time vs final composite, and out-of-sample recession AUC. The out-of-sample number is the one that counts.

In [ ]:
from macro_nowcaster.config import get_settings
from macro_nowcaster.data.fred_client import get_client
from macro_nowcaster.backtest.pseudo_realtime import replay, evaluate
from macro_nowcaster.features.transforms import standardized_panel
from macro_nowcaster.models.dfm import fit_pca_factor
s = get_settings(); cl = get_client(s)
rt = replay(cl, s, start='2010-01-01', end='2019-12-31')
raw = {c: cl.get_series(c) for c in s.codes}
final = fit_pca_factor(standardized_panel(raw, s, 'full')).factor
usrec = (cl.get_series('USREC').resample('ME').mean() > 0.5).astype(int)
evaluate(rt, final, usrec)

## From signal to decision: allocation overlay
Replace the synthetic equity series with real returns (e.g. FRED `SP500`) to backtest live.

In [ ]:
import numpy as np, pandas as pd
from macro_nowcaster.backtest.allocation import run_backtest
idx = final.index
eq = pd.Series(np.random.default_rng(0).normal(0.006, 0.04, len(idx)), index=idx)
recp = art.nowcast.prob.reindex(idx).fillna(0)
res = run_backtest(eq, final, recp)
res.metrics

## Fed-text RAG: divergence vs the quant signal
Supply real FOMC statements/minutes text. Without `ANTHROPIC_API_KEY` the retrieval still runs and the analysis is stubbed.

In [ ]:
from macro_nowcaster.llm.fed_rag import FedRAG, FedDocument
docs = [FedDocument('2026-05', 'statement', 'Economic activity has slowed; labor demand cooled; inflation remains elevated.'),
        FedDocument('2026-03', 'minutes', 'Participants noted resilient spending and a tight labor market, favoring patience.')]
rag = FedRAG(docs)
an = rag.analyze(f"Composite {art.summary()['composite']} sd, {art.summary()['regime']} regime")
print('retrieved:', [d.date for d in an.retrieved]); print(an.analysis)